Title: fix_time_format.ipynb

Purpose: 

Author: Onno Nennecke on 04.2025 Modified: 23.04.2025

Input data: 

    - This file lies here: 

Output data:

    - This file lies here: 

In [1]:
import xarray as xr
import Functions.winter_date_func as winter_date_func
import glob
import os
import numpy as np
import cftime

In [2]:
# Check the time format in the netcdf files
'''
path = '/climca/people/onennecke/model_output/'
files = glob.glob(os.path.join(path, '*.nc'))
len(files)
files
for i in files:
    timeseries_ds_set = xr.open_dataset(i)
    # print(i)
    print(timeseries_ds_set['time'].dtype)
'''

"\npath = '/climca/people/onennecke/model_output/'\nfiles = glob.glob(os.path.join(path, '*.nc'))\nlen(files)\nfiles\nfor i in files:\n    timeseries_ds_set = xr.open_dataset(i)\n    # print(i)\n    print(timeseries_ds_set['time'].dtype)\n"

In [2]:
path = '/climca/people/onennecke/model_output/'
# files = glob.glob(os.path.join(path, '*.nc'))
files = sorted(glob.glob(os.path.join(path, '*.nc')))

len(files)
# files

99

In [3]:
ts_ds_in = xr.open_dataset(files[0])
ts_ds_win_in = winter_date_func.add_winter_calendar(ts_ds_in)
ts_win_in = ts_ds_win_in.sel(time=ts_ds_win_in['day_of_winter'].isin(range(1, 183)))


for i in files:
    print(i)
    ts_ds = xr.open_dataset(i)
    ts_ds_win = winter_date_func.add_winter_calendar(ts_ds)
    ts_win = ts_ds_win.sel(time=ts_ds_win['day_of_winter'].isin(range(1, 183)))
    # ts_win = ts_win.assign_coords(old_time=ts_win.time)
    if isinstance(ts_win.time.values[0], cftime.Datetime360Day):
        print('Time format is cftime.Datetime360Day')
        cut = ts_win.isel(time = slice(2, None))
        last_day = ts_win.isel(time=-1)
        long = xr.concat([cut, last_day, last_day], dim='time')
        # Fix day_of_winter and winter_season for the two new padded days
        day_of_winter = long['day_of_winter'].values
        day_of_winter[-2] = 91
        day_of_winter[-1] = 92

        winter_year = long['winter_year'].values
        winter_season = np.array([f"{y}-{d:03d}" for y, d in zip(winter_year, day_of_winter)])

        # Reassign corrected coordinates
        ts_win = long.assign_coords(
            day_of_winter=('time', day_of_winter),
            winter_season=('time', winter_season)
        )
    if ts_win.time.dtype != 'datetime64[ns]':
        print('Time format is not datetime64[ns]')
        ts_win = ts_win.assign_coords({'time': ('time', ts_win_in.time.values)})
    print(len(ts_win.sel(time=ts_win['winter_year']==2014).time.values))
    ts_win.to_netcdf('/climca/people/onennecke/model_output/winter_data/' + i[38:-3] + '_win.nc')

# print(ts_win.ESM_run.values)
# ts_win

/climca/people/onennecke/model_output/ACCESS-CM2_r1i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/ACCESS-CM2_r4i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/ACCESS-CM2_r5i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/BCC-CSM2-MR_r1i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r10i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r11i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r4i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/EC-Earth3_r101i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r102i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r103i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r104i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3

In [4]:
ts_win

<xarray.Dataset> Size: 219kB
Dimensions:        (time: 1820)
Coordinates:
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    country        float64 8B 9.0
    period         <U4 16B 'week'
    run            <U8 32B 'r9i1p1f2'
    ESM            <U11 44B 'UKESM1-0-LL'
    ESM_run        <U20 80B 'UKESM1-0-LL_r9i1p1f2'
    winter_year    (time) int64 15kB 2014 2014 2014 2014 ... 2024 2024 2024 2024
    day_of_winter  (time) int64 15kB 93 94 95 96 97 98 99 ... 87 88 89 90 91 92
    winter_season  (time) <U8 58kB '2014-093' '2014-094' ... '2024-092'
  * time           (time) datetime64[ns] 15kB 2015-01-01T12:00:00 ... 2024-12...
Data variables:
    temp           (time) float64 15kB 4.046 2.714 3.741 ... 3.884 3.884 3.884
    demand         (time) float64 15kB 1.47e+03 1.486e+03 ... 1.472e+03
    wind_off_prod  (time) float64 15kB 131.1 61.58 71.29 ... 197.7 197.7 197.7
    wind_on_prod   (time) float64 15kB 178.9 98.0 134.2 ... 1.15e+03 1.15e+03
    solar_prod     (time) float64 15kB 87.99 103.8 101.9 ... 58.22 58.22 58.22
    total_prod     (time) float64 15kB 397.9 263.4 307.4 ... 1.405e+03 1.405e+03
    Netto          (time) float64 15kB -1.072e+03 -1.223e+03 ... -66.37 -66.37
    Residual_load  (time) float64 15kB 1.072e+03 1.223e+03 ... 66.37 66.37

In [5]:
# Old code without clipping front and appending the end
ts_ds_in = xr.open_dataset(files[0])
ts_ds_win_in = winter_date_func.add_winter_calendar(ts_ds_in)
ts_win_in = ts_ds_win_in.sel(time=ts_ds_win_in['day_of_winter'].isin(range(1, 183)))


for i in files:
    print(i)
    ts_ds = xr.open_dataset(i)
    ts_ds_win = winter_date_func.add_winter_calendar(ts_ds)
    ts_win = ts_ds_win.sel(time=ts_ds_win['day_of_winter'].isin(range(1, 183)))
    # ts_win = ts_win.assign_coords(old_time=ts_win.time)
    if ts_win.time.dtype != 'datetime64[ns]':
        ts_win = ts_win.assign_coords({'time': ('time', ts_win_in.time.values)})
    print(len(ts_win.sel(time=ts_win['winter_year']==2014).time.values))
    # ts_win.to_netcdf('/climca/people/onennecke/model_output/winter_data/' + i[38:-3] + '_win.nc')

# print(ts_win.ESM_run.values)
# ts_win

/climca/people/onennecke/model_output/ACCESS-CM2_r1i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/ACCESS-CM2_r4i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/ACCESS-CM2_r5i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/BCC-CSM2-MR_r1i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r10i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r11i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/CESM2_r4i1p1f1_timeseries.nc
Time format is not datetime64[ns]
90
/climca/people/onennecke/model_output/EC-Earth3_r101i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r102i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r103i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3_r104i1p1f1_timeseries.nc
90
/climca/people/onennecke/model_output/EC-Earth3

In [6]:
import cftime
from datetime import datetime, timedelta

days = 181

# Define the starting date in the format DD.MM.
for year in range(2014, 2025):
    # Normal calendar (using datetime)
    date_normal = datetime(year, 10, 1)
    resulting_date_normal = date_normal + timedelta(days=days)
    
    # DatetimeNoLeap (365-day year)
    date_no_leap = cftime.DatetimeNoLeap(year, 10, 1)
    resulting_date_no_leap = date_no_leap + timedelta(days=days)
    
    # Datetime360Day (360-day year)
    date_360_day = cftime.Datetime360Day(year, 10, 1)
    resulting_date_360_day = date_360_day + timedelta(days=days)

    print(f"Year: {year}")
    print(f"  Normal Calendar Result:   {resulting_date_normal.strftime('%d.%m.%Y')}")
    print(f"  No Leap Year Result:      {resulting_date_no_leap.strftime('%d.%m.%Y')}")
    print(f"  360-Day Calendar Result:  {resulting_date_360_day.strftime('%d.%m.%Y')}")
    print('-' * 50)


Year: 2014
  Normal Calendar Result:   31.03.2015
  No Leap Year Result:      31.03.2015
  360-Day Calendar Result:  02.04.2015
--------------------------------------------------
Year: 2015
  Normal Calendar Result:   30.03.2016
  No Leap Year Result:      31.03.2016
  360-Day Calendar Result:  02.04.2016
--------------------------------------------------
Year: 2016
  Normal Calendar Result:   31.03.2017
  No Leap Year Result:      31.03.2017
  360-Day Calendar Result:  02.04.2017
--------------------------------------------------
Year: 2017
  Normal Calendar Result:   31.03.2018
  No Leap Year Result:      31.03.2018
  360-Day Calendar Result:  02.04.2018
--------------------------------------------------
Year: 2018
  Normal Calendar Result:   31.03.2019
  No Leap Year Result:      31.03.2019
  360-Day Calendar Result:  02.04.2019
--------------------------------------------------
Year: 2019
  Normal Calendar Result:   30.03.2020
  No Leap Year Result:      31.03.2020
  360-Day Calend